In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn import tree
from sklearn import model_selection
from sklearn import linear_model #линейные модели
from sklearn import metrics
from sklearn import ensemble


In [ ]:
dt = pd.read_csv('wineQualityReds.csv')
dt.shape

(1599, 13)

In [18]:

dt.head()

,Unnamed: 0,fixed.acidity,volatile.acidity,citric.acid,residual.sugar,chlorides,free.sulfur.dioxide,total.sulfur.dioxide,density,pH,sulphates,alcohol,quality
0,1,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,2,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,3,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,4,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,5,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [19]:
dt = dt.drop_duplicates()

In [20]:
dt['range'] = dt['quality'].apply(lambda x: 1 if x>=6 else 0)
dt = dt.drop(['Unnamed: 0', 'quality'], axis=1)

In [21]:
print(dt.corr()['range'].sort_values(ascending=False))

range                   1.000000
alcohol                 0.434751
sulphates               0.218072
citric.acid             0.159129
fixed.acidity           0.095093
residual.sugar         -0.002160
pH                     -0.003264
free.sulfur.dioxide    -0.061757
chlorides              -0.109494
density                -0.159110
total.sulfur.dioxide   -0.231963
volatile.acidity       -0.321441
Name: range, dtype: float64


In [22]:
#scaler = StandardScaler()
X = dt.drop(['range'], axis=1)
#X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
#X = scaler.fit(X)
y = dt['range']

In [23]:
# Формируем обучающую и тестовую выборки
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.3, random_state=42)#stratify=y, 
print('Train shape: {}'.format(X_train.shape))
print('Test shape: {}'.format(X_test.shape))

Train shape: (1119, 11)
Test shape: (480, 11)


In [24]:
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train class distribution:
 range
1    0.525469
0    0.474531
Name: proportion, dtype: float64
Test class distribution:
 range
1    0.55625
0    0.44375
Name: proportion, dtype: float64


In [25]:
#Создаём объект класса LogisticRegression
log_reg = linear_model.LogisticRegression()
#Обучаем модель, минизируя logloss
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)
#Выводим результирующие коэффициенты
print('F1 score: {:.3f}'.format(metrics.f1_score(y_test, y_pred_log)))
print(metrics.classification_report(y_test, y_pred_log))

F1 score: 0.750
              precision    recall  f1-score   support

           0       0.68      0.71      0.70       213
           1       0.76      0.74      0.75       267

    accuracy                           0.73       480
   macro avg       0.72      0.73      0.72       480
weighted avg       0.73      0.73      0.73       480



c:\Users\LNV\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [26]:
# Инициализируем модель дерева решений с максимальной глубиной 1 и обучаем её
dest = tree.DecisionTreeClassifier(max_depth=10, random_state=42)
dest.fit(X_train, y_train)
y_pred_dest = dest.predict(X_test)
#Выводим результирующие коэффициенты
print('F1 score: {:.3f}'.format(metrics.f1_score(y_test, y_pred_dest)))
print(metrics.classification_report(y_test, y_pred_dest))


F1 score: 0.793
              precision    recall  f1-score   support

           0       0.76      0.68      0.72       213
           1       0.76      0.82      0.79       267

    accuracy                           0.76       480
   macro avg       0.76      0.75      0.75       480
weighted avg       0.76      0.76      0.76       480



In [27]:
# Confusion matrix для логистической
print("Confusion matrix (LogReg):")
print(metrics.confusion_matrix(y_test, y_pred_log))

# Confusion matrix для дерева
print("Confusion matrix (Tree):")
print(metrics.confusion_matrix(y_test, y_pred_dest))

Confusion matrix (LogReg):
[[152  61]
 [ 70 197]]
Confusion matrix (Tree):
[[145  68]
 [ 47 220]]


In [29]:
bag = ensemble.BaggingClassifier(estimator=tree.DecisionTreeClassifier(max_depth=10, random_state=42), n_estimators=1500, random_state=42)
bag.fit(X_train, y_train)
y_pred_bag = bag.predict(X_test)
print('F1 score: {:.3f}'.format(metrics.f1_score(y_test, y_pred_bag)))
print(metrics.classification_report(y_test, y_pred_bag))

F1 score: 0.818
              precision    recall  f1-score   support

           0       0.77      0.79      0.78       213
           1       0.83      0.81      0.82       267

    accuracy                           0.80       480
   macro avg       0.80      0.80      0.80       480
weighted avg       0.80      0.80      0.80       480

